In [1]:
import os
import pandas as pd
from pathlib import Path
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Karasev et al.)

This notebook curates supplementary datasets from **Karasev et al.** and produces two standardized outputs: (i) a hemolysis **regression** dataset containing continuous hemolysis measurements, and (ii) an additional set of sequences extracted from a reference table with no explicit hemolysis class labels. Both components are cleaned, duplicate-checked, and exported together with metadata.

- **Toxic effect / endpoint:** hemolytic
- **Source:** Karasev et al.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads two supplementary Excel tables**:
  - `mmc3`: contains peptide sequences with experimental measurements, including `Hemolysis (%)` and `Concentration (µM)`.
  - `mmc5`: contains peptide sequences and a reference field, but no explicit labels.
- **Builds a regression dataset** from `mmc3`:
  - standardizes column names (e.g., `Hemolysis (%)` → `percentage_hemolysis`),
  - retains continuous targets for downstream regression modeling.
- **Builds an unlabeled sequence dataset** from `mmc5`:
  - assigns a placeholder `label = 2` to explicitly mark these sequences as **unlabeled/unknown**.
- **Checks duplicated sequences** separately for each component:
  - regression: duplicates are assessed using the measurement column (`percentage_hemolysis`) as the grouping/consistency key,
  - unlabeled: duplicates are collapsed using the placeholder label.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_regression_dataset.csv` (continuous hemolysis target),
  - `detected_unlabel_sequences.csv` (unlabeled sequences),
  - `metadata.json`.

In [2]:
name_source = "Karasev et al."
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_mmc3 = (
    pd.read_excel(f"{PATH_INPUT}/{name_source}/1-s2.0-S2468111324000379-mmc3.xlsx")
    .rename(columns={"Sequence": "sequence", "Hemolysis (%)": "percentage_hemolysis", 
                     "Concentration (µM)": "concentration (µM)"})
)

In [4]:
df_mmc5 = (
    pd.read_excel(f"{PATH_INPUT}/{name_source}/1-s2.0-S2468111324000379-mmc5.xlsx", 
                  skiprows=1, names=["sequence", "reference"])
    .assign(label=2) # There is no information about the labels of this source, therefore it will be identified with a 2
    [["sequence", "label"]]
)

- Checking duplicates

In [5]:
df_remove_duplicated_reg, df_errors_reg, df_unique_reg = processing_duplicated(df_mmc3, group_seq="sequence", sort_key="percentage_hemolysis")
df_full_reg = pd.concat([df_unique_reg, df_remove_duplicated_reg], axis=0)

In [6]:
df_remove_duplicated_class, df_errors_class, df_unique_class = processing_duplicated(df_mmc5, group_seq="sequence", sort_key="label")
df_full_class = pd.concat([df_unique_class, df_remove_duplicated_class], axis=0)

In [7]:
df_full = pd.concat([df_full_reg, df_full_class])

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
raw_total_sequences = (len(df_mmc5) + len(df_mmc3))

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_unlabel_sequences"  : len(df_full_class),
    "number_of_regression_sequences" : len(df_full_reg),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 12, 1, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'xlsx',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Percentage of hemolysis;No information',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'Supplementary Material from the Paper',
 'publication': 'https://www.sciencedirect.com/science/article/pii/S2468111324000379',
 'number_of_raw_sequences': 2228,
 'number_of_sequences_retained': 1740,
 'number_of_positive_sequences': 0,
 'number_of_negative_sequences': 0,
 'number_of_unlabel_sequences': 944,
 'number_of_regression_sequences': 796,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full_class.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_unlabel_sequences.csv", index=False)
df_full_reg.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_regression_dataset.csv", index=False)